# ClassRoom: Heavyweight Cloud GPU Inference & Benchmark
### Course Code: 23Z711 — Department of CSE, PSG College of Technology

This Google Colab notebook runs state-of-the-art **heavyweight vision models** on a cloud GPU (NVIDIA T4 / A100):
- **Face Detection**: YOLOv8x-face / YOLOv11x-face (68M parameters)
- **Face Recognition**: InsightFace `buffalo_l` (ResNet-100 / Glint360k, 512-D embeddings)
- **Objective**: Benchmark upper-bound accuracy in dense lecture halls and generate comparative empirical tables.

## 1. Setup Environment & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics insightface onnxruntime-gpu opencv-python tqdm

## 2. Load Models (YOLOv8x + InsightFace ResNet-100)

In [ ]:
import cv2
import torch
import numpy as np
import os
import insightface
from insightface.app import FaceAnalysis
from ultralytics import YOLO

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# 1. Initialize InsightFace buffalo_l (ResNet-100)
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
print("InsightFace ResNet-100 initialized successfully!")

# 2. Initialize YOLOv8x
yolo_model = YOLO('yolov8x.pt')
print("YOLOv8x initialized successfully!")

## 3. Upload Your Classroom Sweep Video
Upload your `sweep.mp4` video using the file browser on the left, or use the upload button below.

In [ ]:
from google.colab import files
print("Please upload your classroom sweep video (e.g., sweep.mp4):")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

## 4. Run Heavyweight Inference & Generate Annotated Video

In [ ]:
cap = cv2.VideoCapture(video_path)
native_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

target_fps = 4.0
step = max(1, int(round(native_fps / target_fps)))

out_video_path = "annotated_classroom_sweep.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_video_path, fourcc, target_fps, (width, height))

total_detections = 0
frame_idx = 0
sampled_count = 0
row_stats = {"Front": 0, "Mid": 0, "Rear": 0}

print(f"Processing sweep: {width}x{height} @ {target_fps} FPS...")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    if frame_idx % step == 0:
        sampled_count += 1
        annotated = frame.copy()

        # Run InsightFace buffalo_l with sensitive 0.25 threshold
        faces = app.get(frame)
        total_detections += len(faces)

        for face in faces:
            if face.det_score < 0.22:
                continue
            b = face.bbox.astype(int)
            x1, y1, x2, y2 = b[0], b[1], b[2], b[3]
            conf = face.det_score

            # Row classification
            if y1 > height * 0.55 or (y2 - y1) > 75:
                row = "Front"
                color = (0, 255, 0)
            elif y1 > height * 0.32 or (y2 - y1) > 38:
                row = "Mid"
                color = (255, 180, 0)
            else:
                row = "Rear"
                color = (0, 140, 255)
            row_stats[row] += 1

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            label = f"{conf:.2f} [{row}]"
            cv2.putText(annotated, label, (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

        # Overlay HUD
        hud = f"Frame {sampled_count} | Faces: {len(faces)} | Total: {total_detections}"
        cv2.rectangle(annotated, (15, 15), (420, 50), (20, 20, 20), -1)
        cv2.putText(annotated, hud, (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        writer.write(annotated)
    frame_idx += 1

cap.release()
writer.release()

print("=" * 60)
print(f"Processing Complete!")
print(f"Total Sampled Frames : {sampled_count}")
print(f"Total Faces Detected : {total_detections}")
print(f"Front Row Faces      : {row_stats['Front']}")
print(f"Middle Row Faces     : {row_stats['Mid']}")
print(f"Rear Row Faces       : {row_stats['Rear']}")
print(f"Annotated Video Saved: {out_video_path}")
print("=" * 60)

## 5. Download the Annotated Video & Findings

In [ ]:
files.download(out_video_path)